In [3]:
import os

from dotenv import load_dotenv

from langchain_community.document_loaders import DirectoryLoader, CSVLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError("OPENAI_API_KEY not found")

print("Environment configured successfully.")

Environment configured successfully.


In [4]:
DATA_PATH = "../data"

loader = DirectoryLoader(
    DATA_PATH,
    glob="**/*.csv",
    loader_cls=CSVLoader,
    loader_kwargs={"encoding": "utf-8"}
)

data = loader.load()

print(f"Documents loaded: {len(data)}")

Documents loaded: 300


In [6]:
# Baseline Chunk
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(data)

print(f"Documents : {len(data)}")
print(f"Chunks    : {len(chunks)}")

Documents : 300
Chunks    : 1538


In [7]:
# create embedding model
embedding_model = OpenAIEmbeddings(
    api_key=os.environ["OPENAI_API_KEY"],
    model="text-embedding-3-small"
)

print("Embedding model initialized.")

Embedding model initialized.


In [8]:
sample_text = chunks[0].page_content

vector = embedding_model.embed_query(sample_text)

print("Text length:", len(sample_text))
print("Vector type:", type(vector))
print("Vector dimensions:", len(vector))
print("First 10 values:", vector[:10])

Text length: 257
Vector type: <class 'list'>
Vector dimensions: 1536
First 10 values: [-0.011566162109375, -0.042510986328125, 0.045135498046875, 0.06439208984375, -0.0164031982421875, -0.017852783203125, 0.01172637939453125, 0.04193115234375, 0.0233154296875, 0.003025054931640625]


In [9]:
# Create ChromaDB
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    collection_name="CSV_RAG_BASELINE",
    persist_directory="../chroma_db"
)

print("Chroma vector store created successfully.")
print(f"Stored {len(chunks)} chunks.")

Chroma vector store created successfully.
Stored 1538 chunks.


In [10]:
# First Semantic Search
question = "Which product is associated with Renal Cell Carcinoma?"

In [11]:
results = vector_store.similarity_search_with_relevance_scores(
    question,
    k=3
)

for i, (doc, score) in enumerate(results, start=1):

    print("=" * 80)
    print(f"Result {i}")
    print(f"Relevance Score: {score}")
    print(f"Metadata: {doc.metadata}")
    print(doc.page_content)

Result 1
Relevance Score: 0.36423548431578245
Metadata: {'source': '..\\data\\Pharma_Sales_Long.csv', 'row': 85}
Notes: Product Overview: WELIREG is used in Renal Cell Carcinoma. Clinical discussion covered approved indications, patient eligibility, efficacy, safety, monitoring, treatment pathway, and physician education. Sales discussion included customer questions, objections, market access, competitor comparison, follow-up planning, CRM updates, scientific literature sharing, compliant promotion, and future engagement opportunities. Territory insights included prescription trends, customer behavior,
Result 2
Relevance Score: 0.36423548431578245
Metadata: {'source': '..\\data\\Pharma_Sales_Long.csv', 'row': 135}
Notes: Product Overview: WELIREG is used in Renal Cell Carcinoma. Clinical discussion covered approved indications, patient eligibility, efficacy, safety, monitoring, treatment pathway, and physician education. Sales discussion included customer questions, objections, market 

In [12]:
embedding_result = {
    "Experiment_ID": "E001",
    "Area": "Embedding",
    "Configuration": "OpenAI text-embedding-3-small",
    "Chunk_Size": 500,
    "Chunk_Overlap": 50,
    "Chunks": len(chunks),
    "Question": question,
    "Observation": "Baseline embedding configuration",
    "Conclusion": "Used as baseline for subsequent retrieval experiments"
}

print(embedding_result)

{'Experiment_ID': 'E001', 'Area': 'Embedding', 'Configuration': 'OpenAI text-embedding-3-small', 'Chunk_Size': 500, 'Chunk_Overlap': 50, 'Chunks': 1538, 'Question': 'Which product is associated with Renal Cell Carcinoma?', 'Observation': 'Baseline embedding configuration', 'Conclusion': 'Used as baseline for subsequent retrieval experiments'}
